### Step 3 - Novelty Scoring (Unsupervised, Temporal) - Feature Construction Stage

In [23]:
import pandas as pd
import numpy as np
from tqdm import tqdm

- STEP 3B — Citation Features
- Extract:
- cited_by_count
- reference_count

In [5]:
meta = pd.read_csv("../../outputs/openalex_metadata_full.csv")

citation_df = meta[[
    "global_paper_id",
    "cited_by_count",
    "referenced_works",
    "year"
]].copy()


def count_refs(x):
    if pd.isna(x):
        return 0
    try:
        return len(eval(x))
    except:
        return 0


citation_df["reference_count"] = citation_df["referenced_works"].apply(count_refs)

citation_df = citation_df.rename(columns={
    "global_paper_id": "paper_id"
})

citation_df = citation_df[[
    "paper_id",
    "year",
    "cited_by_count",
    "reference_count"
]]

citation_df.to_csv("../../outputs/citation_features.csv", index=False)

print("Citation features saved.")

Citation features saved.


- STEP 3C — Feature Matrix Construction
- Combine:
- Semantic novelty (kNN)
- Structural novelty
- Citation features

In [6]:
semantic_df = pd.read_csv("../../outputs/semantic_novelty_knn_scores.csv")
struct_df = pd.read_csv("../../outputs/structural_novelty_scores.csv")
citation_df = pd.read_csv("../../outputs/citation_features.csv")

# Merge
features = semantic_df.merge(struct_df,
                             on="paper_id",
                             how="left")

features = features.merge(citation_df,
                          on=["paper_id", "year"],
                          how="left")

features = features.fillna(0)

print("Feature matrix shape:", features.shape)

features.to_csv("../../outputs/novelty_feature_matrix.csv", index=False)

Feature matrix shape: (2511, 7)


- STEP 3D — Composite Novelty Score (Continuous)
- Instead of classification, we define:
-   composite_novelty =
-       w1 * structural +
-       w2 * semantic +
-       w3 * citation_signal
- Citation signal normalized.

In [7]:
features = pd.read_csv("../../outputs/novelty_feature_matrix.csv")

# Normalize citation signal (log transform for stability)
features["citation_signal"] = np.log1p(features["cited_by_count"])


# Min-max normalize components
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


features["struct_norm"] = minmax(features["structural_novelty"])
features["semantic_norm"] = minmax(features["semantic_novelty"])
features["citation_norm"] = minmax(features["citation_signal"])

# Weighted combination (can tune later)
features["composite_novelty"] = (
        0.5 * features["struct_norm"] +
        0.4 * features["semantic_norm"] +
        0.1 * features["citation_norm"]
)

features.to_csv("../../outputs/novelty_feature_matrix_with_score.csv",
                index=False)

print("Composite novelty score computed.")

Composite novelty score computed.


In [20]:
pd.read_csv('../../outputs/novelty_feature_matrix_with_score.csv')['composite_novelty']

0       0.918773
1       0.915936
2       0.883892
3       0.882566
4       0.905863
          ...   
2506    0.038882
2507    0.046332
2508    0.035390
2509    0.045565
2510    0.175166
Name: composite_novelty, Length: 2511, dtype: float64